In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.morse_dataset import MorseDataset
from models.crnn import MorseCRNN1D
import optuna
from src.train_model import train_model
from src.collate_fn import collate_fn, test_collate_fn
from src.encoding_decoding import decode_predictions, build_char_idx_dict
from src.evaluate import decode_ctc_batch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
df_train_full = pd.read_csv("data/train/labels.csv")
df_train_full["filename"] = df_train_full["filename"].map(lambda x: x.replace(".wav", ".pt"))
df_train, df_val = train_test_split(df_train_full, test_size=0.2, random_state=42)
df_train

,filename,text
21753,21753.pt,197
251,00251.pt,86
22941,22941.pt,94176
618,00618.pt,9535
17090,17090.pt,959
...,...,...
29802,29802.pt,07-8860
5390,05390.pt,039638
860,00860.pt,28-7
15795,15795.pt,5448374


In [3]:
alphabet = "0123456789- "
char_to_idx, idx_to_char = build_char_idx_dict(alphabet=alphabet)

train_files = ["data/train_specs_2/" + str(f) for f in df_train["filename"].values]
train_labels = df_train["text"].astype(str).values.tolist()

val_files = ["data/train_specs_2/" + str(f) for f in df_val["filename"].values]
val_labels = df_val["text"].astype(str).values.tolist()

train_dataset = MorseDataset(train_files, train_labels, char_to_idx)
val_dataset = MorseDataset(val_files, val_labels, char_to_idx)

In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

In [5]:
num_classes = len(char_to_idx)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)

def objective(trial):
    channels_configs = [
        [64, 128],
        [64, 128, 256],
        [64, 128, 256, 512],
        [32, 64, 128],
        [128, 256],
    ]
    config_idx = trial.suggest_categorical('config_idx', range(len(channels_configs)))
    
    params = {
        'kernel_size': trial.suggest_int('kernel_size', 3, 13, step=2),
        'hidden_size': trial.suggest_int('hidden_size', 32, 192, step=16),
        'num_layers': trial.suggest_int('num_layers', 2, 5),
        'dropout': trial.suggest_float('dropout', 0.1, 0.4),
        'channels': channels_configs[config_idx]
    }
    
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    grad_clip = trial.suggest_float('grad_clip', 1.0, 5.0, log=True)
    
    model = MorseCRNN1D(num_classes=num_classes, model_type='gru', **params)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )
    model, history = train_model(
        model,
        criterion,
        optimizer,
        train_loader,
        val_loader,
        epochs=15,
        device=device,
        scheduler=scheduler,
        save_by_leven=True,
        idx_to_char=idx_to_char,
        grad_clip=grad_clip,
        save_path=f'optuna_gru/morse_gru_{trial.number}',
    )
    
    return min(history['val_leven'])

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15)
print("Лучшие параметры: ", study.best_params)

[I 2026-06-03 15:19:56,923] A new study created in memory with name: no-name-8b3f5045-2bce-4202-98b2-7a63b63d32b8


Epoch 1/15 | train_loss: 5.5635 | val_loss: 3.2224 | best_val_loss: 3.2224 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 2/15 | train_loss: 2.7415 | val_loss: 2.5517 | best_val_loss: 3.2224 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 3/15 | train_loss: 2.3635 | val_loss: 2.3318 | best_val_loss: 3.2224 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 4/15 | train_loss: 2.0067 | val_loss: 1.7014 | best_val_loss: 1.7014 | avg_leven_dist: 4.0760 | best_leven_dist: 4.0760
Epoch 5/15 | train_loss: 1.4862 | val_loss: 0.9498 | best_val_loss: 0.9498 | avg_leven_dist: 1.4783 | best_leven_dist: 1.4783
Epoch 6/15 | train_loss: 1.0338 | val_loss: 0.8598 | best_val_loss: 0.8598 | avg_leven_dist: 1.2768 | best_leven_dist: 1.2768
Epoch 7/15 | train_loss: 0.8423 | val_loss: 0.7651 | best_val_loss: 0.7651 | avg_leven_dist: 1.2023 | best_leven_dist: 1.2023
Epoch 8/15 | train_loss: 0.7299 | val_loss: 0.5309 | best_val_loss: 0.5309 | avg_leven_dist: 0.8022 | best_leven_dist:

[I 2026-06-03 15:30:31,292] Trial 0 finished with value: 0.5033333333333333 and parameters: {'config_idx': 1, 'kernel_size': 11, 'hidden_size': 64, 'num_layers': 5, 'dropout': 0.2937027604410803, 'lr': 0.00028182921326027195, 'weight_decay': 1.823645075203618e-05, 'grad_clip': 2.102259981432374}. Best is trial 0 with value: 0.5033333333333333.


Epoch 15/15 | train_loss: 0.4157 | val_loss: 0.3267 | best_val_loss: 0.3231 | avg_leven_dist: 0.5033 | best_leven_dist: 0.5033
Epoch 1/15 | train_loss: 132.5136 | val_loss: 84.1912 | best_val_loss: 84.1912 | avg_leven_dist: 6.9023 | best_leven_dist: 6.9023
Epoch 2/15 | train_loss: 58.8424 | val_loss: 29.3370 | best_val_loss: 29.3370 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 3/15 | train_loss: 24.0578 | val_loss: 12.3003 | best_val_loss: 29.3370 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 4/15 | train_loss: 11.3109 | val_loss: 6.7164 | best_val_loss: 29.3370 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 5/15 | train_loss: 6.4891 | val_loss: 4.5432 | best_val_loss: 29.3370 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 6/15 | train_loss: 4.4686 | val_loss: 3.5988 | best_val_loss: 29.3370 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 7/15 | train_loss: 3.5579 | val_loss: 3.1817 | best_val_loss: 29.3370 | avg_leven_dist: 4.5272 | 

[I 2026-06-03 15:38:20,202] Trial 1 finished with value: 4.527166666666667 and parameters: {'config_idx': 3, 'kernel_size': 3, 'hidden_size': 32, 'num_layers': 2, 'dropout': 0.2849909078250169, 'lr': 1.1800439311855474e-05, 'weight_decay': 2.1927396266013482e-06, 'grad_clip': 2.8027367010650432}. Best is trial 0 with value: 0.5033333333333333.


...

Epoch 15/15 | train_loss: 0.2401 | val_loss: 0.1740 | best_val_loss: 0.1740 | avg_leven_dist: 0.2617 | best_leven_dist: 0.2617
Epoch 1/15 | train_loss: 4.9146 | val_loss: 2.7972 | best_val_loss: 2.7972 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 2/15 | train_loss: 2.3788 | val_loss: 1.5224 | best_val_loss: 2.7972 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 3/15 | train_loss: 0.8407 | val_loss: 0.5304 | best_val_loss: 0.5304 | avg_leven_dist: 0.8232 | best_leven_dist: 0.8232
Epoch 4/15 | train_loss: 0.5710 | val_loss: 0.4425 | best_val_loss: 0.4425 | avg_leven_dist: 0.6420 | best_leven_dist: 0.6420
Epoch 5/15 | train_loss: 0.4517 | val_loss: 0.3636 | best_val_loss: 0.3636 | avg_leven_dist: 0.5450 | best_leven_dist: 0.5450
Epoch 6/15 | train_loss: 0.3648 | val_loss: 0.2618 | best_val_loss: 0.2618 | avg_leven_dist: 0.3803 | best_leven_dist: 0.3803
Epoch 7/15 | train_loss: 0.3392 | val_loss: 0.2501 | best_val_loss: 0.2501 | avg_leven_dist: 0.3680 | best_leven_dist

[I 2026-06-03 20:14:43,707] Trial 13 finished with value: 0.23866666666666667 and parameters: {'config_idx': 2, 'kernel_size': 3, 'hidden_size': 160, 'num_layers': 4, 'dropout': 0.19341234660932027, 'lr': 0.0009691704525493142, 'weight_decay': 5.082514622705047e-06, 'grad_clip': 1.0077775988477167}. Best is trial 13 with value: 0.23866666666666667.


Epoch 15/15 | train_loss: 0.1765 | val_loss: 0.1681 | best_val_loss: 0.1556 | avg_leven_dist: 0.2528 | best_leven_dist: 0.2387
Epoch 1/15 | train_loss: 5.8463 | val_loss: 3.0248 | best_val_loss: 3.0248 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 2/15 | train_loss: 2.4335 | val_loss: 1.7387 | best_val_loss: 3.0248 | avg_leven_dist: 4.5272 | best_leven_dist: 4.5272
Epoch 3/15 | train_loss: 1.0773 | val_loss: 0.4979 | best_val_loss: 0.4979 | avg_leven_dist: 0.7205 | best_leven_dist: 0.7205
Epoch 4/15 | train_loss: 0.5082 | val_loss: 0.3666 | best_val_loss: 0.3666 | avg_leven_dist: 0.4925 | best_leven_dist: 0.4925
Epoch 5/15 | train_loss: 0.4201 | val_loss: 0.2617 | best_val_loss: 0.2617 | avg_leven_dist: 0.3938 | best_leven_dist: 0.3938
Epoch 6/15 | train_loss: 0.3738 | val_loss: 0.2819 | best_val_loss: 0.2617 | avg_leven_dist: 0.4393 | best_leven_dist: 0.3938
Epoch 7/15 | train_loss: 0.3598 | val_loss: 0.3497 | best_val_loss: 0.2617 | avg_leven_dist: 0.4482 | best_leven_dist

[I 2026-06-03 20:41:11,010] Trial 14 finished with value: 0.2615 and parameters: {'config_idx': 2, 'kernel_size': 3, 'hidden_size': 112, 'num_layers': 4, 'dropout': 0.18387515013334524, 'lr': 0.00048548282667395067, 'weight_decay': 6.1040932924836655e-06, 'grad_clip': 1.0039902454518954}. Best is trial 13 with value: 0.23866666666666667.


Epoch 15/15 | train_loss: 0.2321 | val_loss: 0.1961 | best_val_loss: 0.1756 | avg_leven_dist: 0.2980 | best_leven_dist: 0.2615
Лучшие параметры:  {'config_idx': 2, 'kernel_size': 3, 'hidden_size': 160, 'num_layers': 4, 'dropout': 0.19341234660932027, 'lr': 0.0009691704525493142, 'weight_decay': 5.082514622705047e-06, 'grad_clip': 1.0077775988477167}


In [ ]:
#best = study.best_params

channels_configs = [
    [64, 128],
    [64, 128, 256],
    [64, 128, 256, 512],
    [32, 64, 128],
    [128, 256],
]

best_params = {
    "kernel_size": best["kernel_size"],
    "hidden_size": best["hidden_size"],
    "num_layers": best["num_layers"],
    "dropout": best["dropout"],
    "channels": channels_configs[best["config_idx"]],
}

model = MorseCRNN1D(
    num_classes=num_classes,
    model_type="gru",
    **best_params,
).to(device)

best_model_path = "optuna_gru/morse_gru_13_12_best.pt"
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

num_classes = len(char_to_idx)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.AdamW(model.parameters(), lr=best['lr'], weight_decay=best['weight_decay'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

model, history = train_model(
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    epochs=20,
    device=device,
    scheduler=scheduler,
    idx_to_char=idx_to_char,
    grad_clip=best['grad_clip'],
    save_path="morse_gru_optuna_train"
)

NameError: name 'study' is not defined

In [6]:
submission_df = pd.read_csv("submits/sample_submission.csv")
submission_df["filename"] = submission_df["filename"].map(lambda x: x.replace(".wav", ".pt"))

test_files = ["data/test_specs_2/" + str(f) for f in submission_df["filename"].values]
test_labels = [""] * len(test_files)

test_dataset = MorseDataset(test_files, test_labels, char_to_idx)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=test_collate_fn,
)

In [10]:
best = {'config_idx': 2, 'kernel_size': 3, 'hidden_size': 160, 'num_layers': 4, 'dropout': 0.19341234660932027, 'lr': 0.0009691704525493142, 'weight_decay': 5.082514622705047e-06, 'grad_clip': 1.0077775988477167}
channels_configs = [
    [64, 128],
    [64, 128, 256],
    [64, 128, 256, 512],
    [32, 64, 128],
    [128, 256],
]

num_classes = len(char_to_idx)
model = MorseCRNN1D(
    num_classes=num_classes,
    model_type='gru',
    kernel_size=best['kernel_size'],
    hidden_size=best['hidden_size'],
    channels=channels_configs[best['config_idx']],
    num_layers=best['num_layers'],
    dropout=best['dropout'],
    bidirectional=True
).to(device)
best_model_path = "best_model/morse_gru_optuna_train_best_every.pt"
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

pred_texts = []

with torch.no_grad():
    for inputs, input_lengths in test_loader:
        inputs = inputs.to(device)

        outputs = model(inputs)

        output_time = outputs.size(0) if outputs.size(1) == inputs.size(0) else outputs.size(1)
        input_time = inputs.size(-1)

        scale = output_time / input_time
        output_lengths = torch.floor(input_lengths.float() * scale).long()
        output_lengths = output_lengths.clamp(min=1, max=output_time)
        
        if outputs.size(0) == inputs.size(0):
            outputs = outputs.transpose(0, 1)

        batch_texts = decode_ctc_batch(
            outputs=outputs,
            output_lengths=output_lengths,
            idx_to_char=idx_to_char
        )
        
        pred_texts.extend(batch_texts)

submission_df["text"] = pred_texts
submission_df["filename"] = submission_df["filename"].map(lambda x: x.replace('.pt', '.wav'))
submission_df.to_csv("submission_final.csv", index=False)

In [11]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Всего параметров: {total_params:,}")

Всего параметров: 2,571,469
